<a href="https://colab.research.google.com/github/NickSchitt2510/uber-forum-analysis/blob/develop/uber_forum_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERT-based Analysis of UberPeople.net Forum Data
This notebook performs advanced analysis of forum discussions to understand:
- Driver resistance strategies against algorithmic management
- Evolution of collective action patterns
- Platform-driver interaction dynamics
## Setup Instructions
Run the following cells to install required packages and initialize the environment.

# Package Installation

In [ ]:
!pip install transformers torch pandas numpy bertopic umap-learn hdbscan plotly tqdm nltk seaborn openpyxl
# umap-learn: for dimensionality reduction
# hdbscan: for clustering

# Import Libraries abd Constant

In [ ]:
# Core data processing
import pandas as pd
import numpy as np
import torch  # For deep learning operations

# BERT and topic modeling
from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from umap import UMAP   # For dimensionality reduction
from hdbscan import HDBSCAN   # For clustering

# Visualization
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# Text processing
import nltk
from nltk.tokenize import sent_tokenize
import re

# Utility
import os
import logging
import warnings
from tqdm.auto import tqdm    # For progress bars
import time

# Mischellous
from sklearn.feature_extraction.text import CountVectorizer
import plotly.graph_objects as go
from google.colab import drive, output
from datetime import datetime
from IPython.display import display, HTML


# Adjust this path to match your Google Drive structure
BASE_FOLDER_PATH = '/content/drive/MyDrive/ColabNotebooks/uber-forum-analysis'
BASE_DATA_PATH = '/content/drive/MyDrive/ColabNotebooks/uber-forum-analysis/data'


# Configure settings
warnings.filterwarnings('ignore')   # Suppress warnings
nltk.download('punkt')              # Download tokenizer data
nltk.download('stopwords')          # Download stopwords
print("Available styles:", plt.style.available)
plt.style.use('seaborn-v0_8')

# plt.style.use('seaborn')
logging.basicConfig(level=logging.INFO)

# Enable dynamic output for Colab
output.enable_custom_widget_manager()

# Check GPU availability and set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


Available styles: ['Solarize_Light2', '_classic_test_patch', '_mpl-gallery', '_mpl-gallery-nogrid', 'bmh', 'classic', 'dark_background', 'fast', 'fivethirtyeight', 'ggplot', 'grayscale', 'petroff10', 'seaborn-v0_8', 'seaborn-v0_8-bright', 'seaborn-v0_8-colorblind', 'seaborn-v0_8-dark', 'seaborn-v0_8-dark-palette', 'seaborn-v0_8-darkgrid', 'seaborn-v0_8-deep', 'seaborn-v0_8-muted', 'seaborn-v0_8-notebook', 'seaborn-v0_8-paper', 'seaborn-v0_8-pastel', 'seaborn-v0_8-poster', 'seaborn-v0_8-talk', 'seaborn-v0_8-ticks', 'seaborn-v0_8-white', 'seaborn-v0_8-whitegrid', 'tableau-colorblind10']
Using device: cuda
GPU Model: Tesla T4
Available GPU memory: 15.83 GB


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Use Google Colab's GPU

(which is much faster than CPU), first you need to enable GPU runtime in Colab. Here's how to do it:

First, Enable GPU in Colab:

1. Click on "Runtime" in the top menu
2. Select "Change runtime type"
3. In the dropdown menu for "Hardware accelerator", select "GPU"
4. Click "Save"

Then verify GPU is available and check its specifications:

In [ ]:
# To check GPU

# Check if GPU is available
print("Is GPU available:", torch.cuda.is_available())

# If GPU is available, print its details
if torch.cuda.is_available():
    print("\nGPU Details:")
    print("GPU Model:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())

    # Print GPU memory information
    print("\nGPU Memory Summary:")
    !nvidia-smi

# Constant var

In [ ]:
# Adjust this path to match your Google Drive structure
BASE_FOLDER_PATH = '/content/drive/MyDrive/ColabNotebooks/uber-forum-analysis'
BASE_DATA_PATH = '/content/drive/MyDrive/ColabNotebooks/uber-forum-analysis/data'


# Mount Google Drive

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Helper function for text processing

In [ ]:
def clean_text(text):
    """Clean and preprocess text data for BERT/BERTopic:
    lowercase, expand contractions, remove URLs, preserve apostrophes."""

    text = str(text)  # Ensure text is string
    text = re.sub(r'http\S+|www.\S+', '', text)  # Remove URLs
    text = re.sub(r'[^\w\s]', '', text)  # Remove special characters
    text = ' '.join(text.split())  # Remove extra whitespace
    return text.lower()

def save_to_drive(data, filename, folder_path='BERTAnalysis'):
    """Save data to specific folder in Google Drive"""
    full_path = f'{BASE_FOLDER_PATH}/{folder_path}/{filename}'
    try:
        # Create folder if it doesn't exist
        !mkdir -p "{BASE_FOLDER_PATH}/{folder_path}"

        # Save based on file type
        if isinstance(data, pd.DataFrame):
            data.to_csv(full_path)
        elif isinstance(data, dict):
            np.save(full_path, data)
        elif isinstance(data, plt.Figure):
            data.savefig(full_path)
        else:
            with open(full_path, 'wb') as f:
                np.save(f, data)
        print(f"Successfully saved {filename} to Drive")
    except Exception as e:
        print(f"Error saving {filename}: {str(e)}")

def display_status(message, status="info"):
    """
    Display status messages with colored formatting
    """
    colors = {
        "info": "blue",
        "success": "green",
        "warning": "orange",
        "error": "red"
    }
    display(HTML(f'<div style="color: {colors[status]};">{message}</div>'))

# Helper Functions for Memory Management

To manage memory usage in Google Colab and prevent RAM limitations

In [ ]:
import gc
import torch
import psutil
import numpy as np
from IPython.display import display, HTML

def clear_memory():
    """Clear memory and cache"""
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

def get_memory_usage():
    """Get current memory usage"""
    process = psutil.Process()
    ram_usage = process.memory_info().rss / 1024 / 1024  # Convert to MB
    if torch.cuda.is_available():
        gpu_usage = torch.cuda.memory_allocated() / 1024 / 1024  # Convert to MB
        return ram_usage, gpu_usage
    return ram_usage, 0

def monitor_memory(func):
    """Decorator to monitor memory usage"""
    def wrapper(*args, **kwargs):
        ram_before, gpu_before = get_memory_usage()
        result = func(*args, **kwargs)
        ram_after, gpu_after = get_memory_usage()

        display_status("\nMemory Usage:", "info")
        display_status(f"RAM: {ram_before:.1f}MB → {ram_after:.1f}MB (Δ {ram_after-ram_before:.1f}MB)", "info")
        if gpu_before > 0:
            display_status(f"GPU: {gpu_before:.1f}MB → {gpu_after:.1f}MB (Δ {gpu_after-gpu_before:.1f}MB)", "info")

        return result
    return wrapper

# Data Loading and Combination

In [ ]:
def load_uber_data(base_path):
    """
    Load data from multiple Excel files in year folders

    Parameters:
    base_path: Path to the main data folder containing year folders

    Returns:
    Combined DataFrame of all Excel files
    """

    print(f"Attempting to load data from: {base_path}")

    # First, verify the path exists
    if not os.path.exists(base_path):
        raise Exception(f"Path does not exist: {base_path}")

    # List and print the contents of the directory
    print("\nContents of the directory:")
    try:
        for item in os.listdir(base_path):
            print(f"- {item}")
    except Exception as e:
        print(f"Error listing directory contents: {str(e)}")


    all_data = []  # List to store DataFrames from each file

    # Get list of year folders (assuming they are subdirectories within base_path and are digits)
    year_folders = sorted([f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f)) and f.isdigit()])

    if not year_folders:
        raise Exception(f"No year folders (2014-2025) found in {base_path}")

    print(f"Found {len(year_folders)} potential year folders: {year_folders}")

    # Filter year folders to be within the desired range (2014-2025)
    valid_year_folders = [folder for folder in year_folders if 2014 <= int(folder) <= 2025]
    valid_year_folders.sort() # Ensure chronological order

    if not valid_year_folders:
        raise Exception(f"No year folders found within the range 2014-2025 in {base_path}")

    print(f"Processing {len(valid_year_folders)} year folders within the range 2014-2025: {valid_year_folders}")


    # Iterate through each valid year folder
    for year in tqdm(valid_year_folders, desc="Processing years"):
        year_path = os.path.join(base_path, year)

        # Get all Excel files in the year folder
        excel_files = [f for f in os.listdir(year_path) if f.endswith('.xlsx')]

        print(f"\nProcessing {year}: Found {len(excel_files)} files")

        # Load each Excel file in the year folder
        for file in tqdm(excel_files, desc=f"Loading files from {year}"):
            file_path = os.path.join(year_path, file)
            try:
                # Read the Excel file
                df = pd.read_excel(file_path)

                # Add year and file information
                df['source_year'] = year
                df['source_file'] = file

                all_data.append(df)
                print(f"Loaded {file}: {len(df)} rows")

            except Exception as e:
                print(f"Error loading {file}: {str(e)}")

    # Combine all DataFrames
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print("\nData Loading Summary:")
        print(f"Total files processed: {len(all_data)}")
        print(f"Total rows: {len(combined_df)}")

        # Ensure 'date' column exists and is in datetime format before trying to find min/max
        if 'date' in combined_df.columns and pd.api.types.is_datetime64_any_dtype(combined_df['date']):
             print(f"Date range: {combined_df['date'].min()} to {combined_df['date'].max()}")
        else:
             print("Date range: 'date' column not found or not in datetime format.")


        print("\nColumns in the dataset:")
        for col in combined_df.columns:
            print(f"- {col}: {combined_df[col].dtype}")

        return combined_df
    else:
        raise Exception("No data was loaded!")

# Mount Google Drove
from google.colab import drive
drive.mount('/content/drive')

# Load the data
try:
    # Adjust this path to match your Google Drive structure
    BASE_DATA_PATH = '/content/drive/MyDrive/ColabNotebooks/uber-forum-analysis/data'

    print("Starting data loading process...")
    df = load_uber_data(BASE_DATA_PATH)

    # Basic preprocessing
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['full_text'] = df['title'].fillna('') + ' ' + df['content'].fillna('')
    df['clean_text'] = df['full_text'].apply(clean_text)

    # Display basic information with Colab formatting
    from IPython.display import HTML, display
    import matplotlib.pyplot as plt
    import seaborn as sns
    import re

    # Create summary statistics
    summary_stats = pd.DataFrame({
        'Metric': [
            'Total Posts',
            'Date Range',
            'Unique Authors',
            'Years Covered',
            'Average Posts per Year'
        ],
        'Value': [
            len(df),
            f"{df['date'].min().date() if not df['date'].min() is pd.NaT else 'N/A'} to {df['date'].max().date() if not df['date'].max() is pd.NaT else 'N/A'}",
            df['author'].nunique() if 'author' in df.columns else 'N/A',
            len(df['source_year'].unique()) if 'source_year' in df.columns else 'N/A',
            f"{len(df)/len(df['source_year'].unique()):,.1f}" if 'source_year' in df.columns and len(df['source_year'].unique()) > 0 else 'N/A'

        ]
    })

    print("\nDataset Summary:")
    display(HTML(summary_stats.to_html(index=False)))

    # Save preprocessed data
    save_to_drive(df, 'preprocessed_data.csv')

    # Show distribution of posts across years
    if 'source_year' in df.columns:
        plt.figure(figsize=(12, 6))
        year_counts = df['source_year'].value_counts().sort_index()
        sns.barplot(x=year_counts.index, y=year_counts.values)
        plt.title('Distribution of Posts Across Years')
        plt.xlabel('Year')
        plt.ylabel('Number of Posts')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("\n'source_year' column not found, skipping year distribution plot.")

except Exception as e:
    print(f"Error in data loading process: {str(e)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting data loading process...
Attempting to load data from: /content/drive/MyDrive/ColabNotebooks/uber-forum-analysis/data

Contents of the directory:
- .DS_Store
- 2025
- 2014
- 2019
- 2023
- 2021
- 2015
- 2020
- 2017
- 2024
- 2022
- 2016
- 2018
Found 12 potential year folders: ['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Processing 12 year folders within the range 2014-2025: ['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


Processing years:   0%|          | 0/12 [00:00<?, ?it/s]


Processing 2014: Found 9 files


Loading files from 2014:   0%|          | 0/9 [00:00<?, ?it/s]

Loaded 2014-4.xlsx: 103 rows
Loaded 2014-6.xlsx: 241 rows
Loaded 2014-5.xlsx: 196 rows
Loaded 2014-7.xlsx: 418 rows
Loaded 2014-8.xlsx: 1102 rows
Loaded 2014-9.xlsx: 1309 rows
Loaded 2014-10.xlsx: 1285 rows
Loaded 2014-11.xlsx: 1533 rows
Loaded 2014-12.xlsx: 1624 rows

Processing 2015: Found 12 files


...

Loading files from 2016:   0%|          | 0/12 [00:00<?, ?it/s]

Loaded 2016-1.xlsx: 5594 rows
Loaded 2016-2.xlsx: 4565 rows
Loaded 2016-3.xlsx: 4519 rows
Loaded 2016-4.xlsx: 4689 rows
Loaded 2016-5.xlsx: 5197 rows
Loaded 2016-6.xlsx: 4892 rows
Loaded 2016-7.xlsx: 5083 rows
Loaded 2016-8.xlsx: 5860 rows
Loaded 2016-9.xlsx: 6191 rows
Loaded 2016-10.xlsx: 6331 rows
Loaded 2016-11.xlsx: 5644 rows
Loaded 2016-12.xlsx: 6681 rows

Processing 2017: Found 12 files


Loading files from 2017:   0%|          | 0/12 [00:00<?, ?it/s]

Loaded 2017-1.xlsx: 7468 rows
Loaded 2017-2.xlsx: 6596 rows
Loaded 2017-3.xlsx: 7043 rows
Loaded 2017-4.xlsx: 6934 rows
Loaded 2017-5.xlsx: 7358 rows
Loaded 2017-6.xlsx: 7700 rows
Loaded 2017-7.xlsx: 8454 rows
Loaded 2017-8.xlsx: 7474 rows
Loaded 2017-9.xlsx: 6871 rows
Loaded 2017-10.xlsx: 6444 rows
Loaded 2017-11.xlsx: 6306 rows


KeyboardInterrupt: 

# Initialize BERT Analyzer Class

In [ ]:
class BERTAnalyzer:
    """
    A class for analyzing text data using BERT embeddings and topic modeling.
    This class combines BERT's language understanding capabilities with clustering
    for topic identification.
    """
    def __init__(self, model_name='bert-base-uncased', n_neighbors=15, n_components=5, min_cluster_size=15):
        """
        Initialize the BERT Analyzer with model and components for topic modeling.

        Parameters:
        -----------
        model_name : str
            Name of the BERT model to use (default: 'bert-base-uncased')
            Other options include 'bert-large-uncased', 'bert-base-cased', etc.
        n_neighbors: int
            Number of neighbors to consider for each point
        n_components: int
            Number of dimensions to reduce to
        min_cluster_size: int
            Minimum size of clusters
        """
        # Check for GPU
        if not torch.cuda.is_available():
            print("WARNING: GPU not available, using CPU. This will be slow!")
            print("Go to Runtime > Change runtime type and select GPU")
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")

        # Initialize BERT model components
        print("Loading BERT model and tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)          # Convert text to BERT tokens
        self.model = AutoModel.from_pretrained(model_name).to(self.device)  # Load BERT model

        # Initialize topic modeling components
        # UMAP: Dimensionality reduction technique
        self.umap_model = UMAP(
            n_neighbors=n_neighbors,
            n_components=n_components,
            min_dist=0.0
        )

        # HDBSCAN: Clustering algorithm for topic identification
        self.hdbscan_model = HDBSCAN(
            min_cluster_size=min_cluster_size,
            metric='euclidean'
        )

        # CountVectorizer: Convert text to word frequency vectors
        self.vectorizer = CountVectorizer(
            stop_words="english"  # Remove common English words
        )

        print("Initialization complete!")


    def generate_embeddings(self, texts, batch_size=16, save_path=None):
        """
        Generate BERT embeddings using GPU acceleration

        Parameters:
        -----------
        texts : list
            List of texts to process
        batch_size : int
            Number of texts to process at once (default: 32)
            Increase this if you have enough GPU memory
        save_path : str
            Path to save the generated embeddings (optional)

        Returns:
        --------
        numpy.ndarray
            Array of BERT embeddings
        """
        embeddings = []
        total_texts = len(texts)

        progress_bar = tqdm(total=total_texts, desc="Generating embeddings")

        try:
            for i in range(0, total_texts, batch_size):
                # Get batch of texts
                batch_texts = texts[i:i + batch_size]

                # Memory management: Clear GPU cache periodically
                if i % (batch_size * 10) == 0:
                    if torch.cuda.is_available():
                        clear_memory()

                # Memory management: Clear GPU cache periodically
                if i % (batch_size * 10) == 0:
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                # Convert texts to BERT input format: Tokenize and encode
                encoded = self.tokenizer(
                    batch_texts,
                    padding=True,  # Add padding to make all sequences same length
                    truncation=True,            # Truncate long sequences
                    max_length=512,             # Maximum sequence length
                    return_tensors='pt'         # Return PyTorch tensors
                )

                # Move data to GPU/CPU
                input_ids = encoded['input_ids'].to(self.device)
                attention_mask = encoded['attention_mask'].to(self.device)

                # Generate embeddings without gradient calculation
                with torch.no_grad():
                    outputs = self.model(input_ids, attention_mask=attention_mask)
                    # Get CLS token embedding (first token) for each text
                    batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                    embeddings.extend(batch_embeddings)

                # Update progress
                progress_bar.update(len(batch_texts))


        except RuntimeError as e:
            print(f"\nError: {str(e)}")
            print("\nTrying to recover...")
            torch.cuda.empty_cache()
            raise

        finally:
            progress_bar.close()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return np.array(embeddings)


# Load Saved Embeddings with Progress Monitoring

In [ ]:
# Helper function to load pre-exist embedding file.

def load_embeddings(embedding_path):
    """
    Load pre-generated embeddings with progress bar
    """
    try:
        # Get file size for progress bar
        file_size = os.path.getsize(embedding_path) / 1024**2  # Convert to MB
        display_status(f"Loading embeddings ({file_size:.1f} MB) from: {embedding_path}")

        # Simulate progress for large files (since np.load is atomic)
        with tqdm(total=100, desc="Loading embeddings") as pbar:
            start_time = time.time()
            embeddings = np.load(embedding_path)
            load_time = time.time() - start_time

            # Update progress bar
            pbar.update(100)

        display_status(
            f"✓ Successfully loaded embeddings with shape: {embeddings.shape} in {load_time:.2f}s",
            "success"
        )
        return embeddings

    except Exception as e:
        display_status(f"✕ Error loading embeddings: {str(e)}", "error")
        raise

def verify_embeddings(embeddings, df, show_progress=True):
    """
    Verify embeddings with progress bar
    """
    checks = [
        ("Size Match", lambda: len(embeddings) == len(df)),
        ("Dimension Check", lambda: embeddings.shape[1] == 768),
        ("No NaN Values", lambda: not np.isnan(embeddings).any()),
        ("Valid Range", lambda: np.all(np.abs(embeddings) < 100))
    ]

    display_status("\nVerifying embeddings...", "info")

    results = {}
    with tqdm(total=len(checks), desc="Running checks") as pbar:
        for check_name, check_func in checks:
            try:
                result = check_func()
                results[check_name] = result
                status = "success" if result else "warning"
                display_status(
                    f"{'✓' if result else '⚠️'} {check_name}",
                    status
                )
            except Exception as e:
                results[check_name] = False
                display_status(f"✕ {check_name} failed: {str(e)}", "error")

            pbar.update(1)
            time.sleep(0.1)  # Small delay for visibility

    return all(results.values())

In [ ]:
# Generate or Load Embeddings

EMBEDDING_PATH = f'{BASE_FOLDER_PATH}/BERTAnalysis/bert_embeddings.npy'

# Check if embeddings already exist
if os.path.exists(EMBEDDING_PATH):
    display_status("Found existing embeddings file!", "success")
    embeddings = load_embeddings(EMBEDDING_PATH)

    # Optional: Verify loaded embeddings
    if verify_embeddings(embeddings, df):
        display_status("✓ Existing embeddings verified successfully!", "success")
    else:
        display_status("⚠️ Existing embeddings verification failed. Consider regenerating.", "warning")
        # Optionally, you could add logic here to regenerate if verification fails
else:
    display_status("No existing embeddings found. Generating new ones.", "info")
    # Initialize the BERT Analyzer
    analyzer = BERTAnalyzer()

    # Generate embeddings for all cleaned texts
    BATCH_SIZE = 16  # Small batch size to manage memory
    embeddings = analyzer.generate_embeddings(df['clean_text'].tolist(), batch_size=BATCH_SIZE)

    # Save newly generated embeddings
    save_to_drive(embeddings, 'bert_embeddings.npy')
    display_status("✓ New embeddings generated and saved.", "success")

# Ensure 'embeddings' variable is available for subsequent steps
if 'embeddings' in locals():
    print(f"\nEmbeddings ready with shape: {embeddings.shape}")
else:
    display_status("✕ Embeddings could not be loaded or generated.", "error")
    raise RuntimeError("Embeddings not available for further processing.")

Loading embeddings:   0%|          | 0/100 [00:00<?, ?it/s]

NameError: name 'df' is not defined

# Tips for optimal GPU usage

In [ ]:
# Cell to monitor GPU usage during processing
def print_gpu_utilization():
    """Print GPU memory usage"""
    if torch.cuda.is_available():
        print(f"GPU memory used: {torch.cuda.memory_allocated()/1024**2:.2f} MB")
        print(f"GPU memory cached: {torch.cuda.memory_reserved()/1024**2:.2f} MB")
# Function to find optimal batch size
def find_optimal_batch_size(initial_batch_size=32, max_batch_size=128, step=32):
    """
    Find the largest batch size that works with your GPU memory
    """
    if not torch.cuda.is_available():
        print("GPU not available!")
        return initial_batch_size

    batch_size = initial_batch_size
    while batch_size <= max_batch_size:
        try:
            # Try processing a single batch
            sample_texts = df['clean_text'].tolist()[:batch_size]
            analyzer.generate_embeddings(sample_texts, batch_size=batch_size)
            print(f"Batch size {batch_size} works!")
            batch_size += step
        except RuntimeError as e:
            print(f"Batch size {batch_size} is too large!")
            return batch_size - step

    return batch_size
# Try to find optimal batch size
print("Finding optimal batch size...")
BATCH_SIZE = find_optimal_batch_size()
print(f"Optimal batch size: {BATCH_SIZE}")

# Load the preprocessed data

In [ ]:
def load_preprocessed_data(file_path, chunk_size=10000):
    """
    Load preprocessed data with memory management

    Parameters:
    -----------
    file_path : str
        Path to the preprocessed CSV file
    chunk_size : int
        Size of chunks to process at once

    Returns:
    --------
    pandas.DataFrame
        Loaded and processed dataframe
    """
    def clear_memory():
        """Clear memory and cache"""
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    def get_memory_usage():
        """Get current memory usage in MB"""
        process = psutil.Process()
        ram_usage = process.memory_info().rss / 1024 / 1024
        gpu_usage = torch.cuda.memory_allocated() / 1024 / 1024 if torch.cuda.is_available() else 0
        return ram_usage, gpu_usage
    try:
        # Import required libraries
        import gc
        import psutil
        import torch
        from tqdm.notebook import tqdm

        # Initial memory check
        initial_ram, initial_gpu = get_memory_usage()
        display_status(f"Initial memory - RAM: {initial_ram:.1f}MB, GPU: {initial_gpu:.1f}MB", "info")

        # Check file existence
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")

        # Get file size
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        display_status(f"File size: {file_size_mb:.1f}MB", "info")

        # Adjust chunk size based on file size
        if file_size_mb > 1000:  # If file is larger than 1GB
            chunk_size = min(chunk_size, int(1000 * 1024 * 1024 / file_size_mb))
            display_status(f"Adjusted chunk size to: {chunk_size} rows", "info")

        # Load data in chunks
        chunks = []
        total_rows = 0

        display_status("Loading data in chunks...", "info")
        for chunk in tqdm(pd.read_csv(file_path, chunksize=chunk_size), desc="Loading chunks"):
            # Process chunk
            if 'clean_text' in chunk.columns:
                chunk['clean_text'] = chunk['clean_text'].astype(str).fillna('')

            chunks.append(chunk)
            total_rows += len(chunk)

            # Monitor memory
            current_ram, current_gpu = get_memory_usage()
            if current_ram > initial_ram * 2:  # If RAM usage doubles
                display_status("High memory usage detected. Combining chunks...", "warning")
                df_temp = pd.concat(chunks, ignore_index=True)
                chunks = [df_temp]
                clear_memory()

            # Show progress
            if total_rows % (chunk_size * 10) == 0:
                display_status(f"Processed {total_rows:,} rows...", "info")

        # Combine all chunks
        display_status("Combining all chunks...", "info")
        df = pd.concat(chunks, ignore_index=True)
        clear_memory()

        # Final memory check
        final_ram, final_gpu = get_memory_usage()
        display_status(f"Final memory - RAM: {final_ram:.1f}MB, GPU: {final_gpu:.1f}MB", "info")

        # Display summary
        info_df = pd.DataFrame({
            'Metric': [
                'Total Posts',
                'Memory Usage (RAM)',
                'Memory Usage (GPU)',
                'Columns'
            ],
            'Value': [
                f"{len(df):,}",
                f"{final_ram:.1f}MB (Δ {final_ram-initial_ram:.1f}MB)",
                f"{final_gpu:.1f}MB (Δ {final_gpu-initial_gpu:.1f}MB)",
                ', '.join(df.columns.tolist())
            ]
        })

        display_status("\nLoaded Data Summary:", "info")
        display(HTML(info_df.to_html(index=False)))

        return df

    except Exception as e:
        display_status(f"Error loading data: {str(e)}", "error")
        clear_memory()
        raise

# Main execution code
try:

    # Clear initial memory
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    PREPROCESSED_DATA_PATH = f'{BASE_FOLDER_PATH}/BERTAnalysis/preprocessed_data.csv'
    display_status(f"Attempting to load preprocessed data from: {PREPROCESSED_DATA_PATH}")

    # Load data with memory management
    df = load_preprocessed_data(PREPROCESSED_DATA_PATH)

    # Verify data integrity
    if 'clean_text' not in df.columns:
        display_status("⚠️ 'clean_text' column not found in loaded data.", "warning")
        raise ValueError("Required 'clean_text' column missing from data")

    # Check for empty or invalid texts
    invalid_texts = df['clean_text'].isna().sum()
    if invalid_texts > 0:
        display_status(f"⚠️ Found {invalid_texts} invalid/empty texts", "warning")

except Exception as e:
    display_status(f"\n✕ Error loading preprocessed data: {str(e)}", "error")
    # Cleanup on error
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    raise

Loading chunks: 0it [00:00, ?it/s]

Metric,Value
Total Posts,"385,149"
Memory Usage (RAM),3810.6MB (Δ 837.8MB)
Memory Usage (GPU),0.0MB (Δ 0.0MB)
Columns,"Unnamed: 0, id, date, title, content, algoactivism_type, author, forum, views, reactions_amount, comments_amount, sentiment, source_year, source_file, full_text, clean_text"


## Save Topic Modeling Result

In [ ]:
def save_topic_modeling_results_by_year(topic_model, topics, probs, df, year_folder, year):
    """
    Save topic modeling results for a specific year with error handling for small datasets

    Parameters:
    -----------
    topic_model : BERTopic model
    topics : array-like
        Topic assignments
    probs : array-like
        Topic probabilities
    df : pandas DataFrame
        Data for the specific year
    year_folder : str
        Path to save results
    year : int or str
        The year being processed
    """
    try:
        # Create year results folder
        os.makedirs(year_folder, exist_ok=True)
        display_status(f"\nSaving results for year {year} to {year_folder}...", "info")

        # 1. Save topic information
        display_status("Saving topic information...", "info")
        topic_info = topic_model.get_topic_info()
        topic_info.to_csv(f'{year_folder}/topic_info.csv')

        # 2. Save topic assignments and probabilities
        display_status("Saving topic assignments and probabilities...", "info")
        topic_data = pd.DataFrame({
            'text': df['clean_text'],
            'topic': topics,
            'date': df['date']
        })
        topic_data.to_csv(f'{year_folder}/topic_assignments.csv')

        # Save probabilities
        prob_columns = [f'Topic_{i}' for i in range(probs.shape[1])]
        prob_df = pd.DataFrame(probs, columns=prob_columns)
        prob_df.to_csv(f'{year_folder}/topic_probabilities.csv')

        # 3. Save topic keywords
        display_status("Saving topic keywords...", "info")
        topic_keywords = {}
        for topic_id, words in topic_model.get_topics().items():
            topic_keywords[topic_id] = [word for word, _ in words[:10]]
        pd.DataFrame.from_dict(topic_keywords, orient='index')\
            .to_csv(f'{year_folder}/topic_keywords.csv')

        # 4. Save visualizations
        display_status("Saving visualizations...", "info")
        clear_memory()

        # Helper function to safely generate visualizations
        def safe_generate_viz(viz_func, filename, viz_name):
            try:
                viz = viz_func()
                viz.write_html(f'{year_folder}/{filename}')
                del viz
                clear_memory()
                return True
            except Exception as e:
                display_status(f"Warning: Could not generate {viz_name} visualization: {str(e)}", "warning")
                return False

        # Try to generate each visualization separately
        visualizations = [
            (topic_model.visualize_topics, 'topic_visualization.html', 'topic network'),
            (topic_model.visualize_hierarchy, 'topic_hierarchy.html', 'topic hierarchy'),
            (topic_model.visualize_heatmap, 'topic_similarity.html', 'topic similarity')
        ]
        viz_success = []
        for viz_func, filename, viz_name in visualizations:
            success = safe_generate_viz(viz_func, filename, viz_name)
            viz_success.append((viz_name, success))

        # Try to generate timeline visualization separately due to its different nature
        try:
            if len(df) > 3:  # Only attempt if we have enough data
                timeline_data = topic_model.topics_over_time(
                    df['clean_text'].tolist(),
                    topics,
                    df['date'].tolist()
                )
                timeline_vis = topic_model.visualize_topics_over_time(timeline_data)
                timeline_vis.write_html(f'{year_folder}/topic_timeline.html')
                del timeline_vis, timeline_data
                clear_memory()
                viz_success.append(('timeline', True))
            else:
                display_status("Warning: Not enough data for timeline visualization", "warning")
                viz_success.append(('timeline', False))
        except Exception as e:
            display_status(f"Warning: Could not generate timeline visualization: {str(e)}", "warning")
            viz_success.append(('timeline', False))

        # 5. Save summary report
        display_status("Saving summary report...", "info")
        with open(f'{year_folder}/analysis_summary.txt', 'w') as f:
            f.write(f"Topic Modeling Analysis Summary for Year {year}\n")
            f.write("============================\n\n")
            f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total Documents: {len(df)}\n")
            f.write(f"Number of Topics: {len(topic_model.get_topics())}\n")
            f.write(f"Date Range: {df['date'].min()} to {df['date'].max()}\n\n")

            f.write("Files Generated:\n")
            f.write("- topic_info.csv: Topic information and metrics\n")
            f.write("- topic_assignments.csv: Document-topic assignments\n")
            f.write("- topic_probabilities.csv: Topic probability distributions\n")
            f.write("- topic_keywords.csv: Top keywords per topic\n")

            f.write("\nVisualization Status:\n")
            for viz_name, success in viz_success:
                status = "Generated" if success else "Failed to generate"
                f.write(f"- {viz_name}: {status}\n")
            f.write("\nTop Topics and Keywords:\n")
            for topic_id in list(topic_model.get_topics().keys())[:5]:
                if topic_id != -1:  # Skip noise topic
                    words = topic_model.get_topic(topic_id)[:5]
                    words_str = ", ".join([word for word, _ in words])
                    f.write(f"Topic {topic_id}: {words_str}\n")
        display_status("✓ Core results saved successfully!", "success")
        if not all(success for _, success in viz_success):
            display_status("⚠️ Some visualizations could not be generated", "warning")
        return True

    except Exception as e:
        display_status(f"Error saving results: {str(e)}", "error")
        return False


# Topic Modeling with Progress Monitoring

In [ ]:
def process_single_year(year, df, embeddings, base_folder_path):
    """
    Process a single year's worth of data with adaptive parameters
    """
    try:
        # Check if year has already been processed
        year_folder = f'{base_folder_path}/BERTAnalysis/year_{year}'
        if os.path.exists(f'{year_folder}/completed.flag'):
            display_status(f"Year {year} has already been processed. Skipping...", "info")
            return True

        # Create year folder
        os.makedirs(year_folder, exist_ok=True)

        # Filter data for this year
        year_mask = df['source_year'] == year
        year_df = df[year_mask].copy()
        year_embeddings = embeddings[year_mask]

        # Get dataset size
        n_documents = len(year_df)
        display_status(f"\nProcessing year {year}", "info")
        display_status(f"Documents for year {year}: {n_documents}", "info")

        # Adjust UMAP parameters based on dataset size
        n_neighbors = min(15, max(2, n_documents - 1))  # Ensure n_neighbors is less than n_samples
        n_components = min(5, max(2, n_documents - 1))  # Adjust number of components

        display_status(f"Using n_neighbors={n_neighbors}, n_components={n_components}", "info")

        # Initialize progress tracking
        total_steps = 3
        current_step = 0

        # Initial memory check
        initial_ram, initial_gpu = get_memory_usage()
        display_status(f"Initial memory usage - RAM: {initial_ram:.1f}MB, GPU: {initial_gpu:.1f}MB", "info")

        with tqdm(total=total_steps, desc=f"Processing Year {year}") as pbar:
            # Step 1: Initialize BERTopic with adjusted parameters
            clear_memory()
            display_status("\nInitializing BERTopic model...", "info")

            # Adjust minimum cluster size based on dataset size
            min_cluster_size = min(15, max(2, n_documents // 10))  # Adjust cluster size

            topic_model = BERTopic(
                umap_model=UMAP(
                    n_neighbors=n_neighbors,
                    n_components=n_components,
                    min_dist=0.0
                ),
                hdbscan_model=HDBSCAN(
                    min_cluster_size=min_cluster_size,
                    metric='euclidean',
                    prediction_data=True
                ),
                vectorizer_model=CountVectorizer(stop_words="english"),
                calculate_probabilities=True
            )
            current_step += 1
            pbar.update(1)

            # Step 2: Fit the model
            clear_memory()
            display_status("\nFitting BERTopic model...", "info")
            start_time = time.time()

            # Check if dataset is too small
            if n_documents < 3:
                display_status(f"Warning: Year {year} has too few documents ({n_documents}) for topic modeling", "warning")
                # Create a simple summary for small datasets
                with open(f'{year_folder}/summary.txt', 'w') as f:
                    f.write(f"Year {year} has insufficient data for topic modeling\n")
                    f.write(f"Number of documents: {n_documents}\n")
                return True

            topics, probs = topic_model.fit_transform(
                year_df['clean_text'].tolist(),
                year_embeddings
            )

            fit_time = time.time() - start_time
            display_status(f"✓ Model fitting completed in {fit_time:.2f}s", "success")
            current_step += 1
            pbar.update(1)

            # Step 3: Save all results using the comprehensive save function
            clear_memory()
            display_status("\nSaving all results...", "info")
            save_success = save_topic_modeling_results_by_year(
                topic_model,
                topics,
                probs,
                year_df,
                year_folder,
                year
            )
            if not save_success:
                display_status("Failed to save results", "error")
                return False
            current_step += 3  # Update progress for the remaining steps
            pbar.update(3)

            # Create completion flag
            with open(f'{year_folder}/completed.flag', 'w') as f:
                f.write(f"Completed: {datetime.now()}")

            display_status(f"\n✓ Year {year} processing completed successfully!", "success")
            return True

    except Exception as e:
        display_status(f"\n✕ Error processing year {year}: {str(e)}", "error")
        display_status("\nTroubleshooting tips:", "info")
        display_status("1. Check the number of documents for this year", "info")
        display_status("2. Verify embedding dimensions", "info")
        display_status("3. Ensure data quality for this year", "info")
        return False

# Function to analyze year before processing
def analyze_year_data(year, df):
    """
    Analyze year data before processing
    """
    year_mask = df['source_year'] == year
    year_df = df[year_mask]

    display_status(f"\nAnalysis for year {year}:", "info")
    display_status(f"Number of documents: {len(year_df)}", "info")
    display_status(f"Number of unique authors: {year_df['author'].nunique()}", "info")
    display_status(f"Average document length: {year_df['clean_text'].str.len().mean():.1f} characters", "info")

    return len(year_df)

# Modified process_specific_year function
def process_specific_year(year, df, embeddings, base_folder_path):
    """
    Process a specific year with pre-analysis
    """
    if year not in df['source_year'].unique():
        display_status(f"Year {year} not found in dataset!", "error")
        return

    # Analyze year data first
    n_docs = analyze_year_data(year, df)

    if n_docs < 3:
        display_status(f"Warning: Year {year} has too few documents ({n_docs}) for topic modeling", "warning")
        return

    display_status(f"\nProceeding with processing year {year}...", "info")
    success = process_single_year(year, df, embeddings, base_folder_path)

    if success:
        display_status(f"Year {year} processed successfully!", "success")
    else:
        display_status(f"Failed to process year {year}", "error")

# Usage example:
try:
    year_to_process = 2014  # Change this to the year you want to process

    # First analyze the year
    analyze_year_data(year_to_process, df)

    # Then process if you want to continue
    process_specific_year(year_to_process, df, embeddings, BASE_FOLDER_PATH)

except Exception as e:
    display_status(f"Error: {str(e)}", "error")
    clear_memory()

Processing Year 2014:   0%|          | 0/3 [00:00<?, ?it/s]